## import lib

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import gradio as gr
from IPython.display import Markdown,display
import sqlite3
import requests
import json

## create ollama client

In [2]:
requests.get("http://localhost:11434").content

b'Ollama is running'

In [12]:
ollama_url="http://localhost:11434/v1"
ollama=OpenAI(base_url=ollama_url,api_key="ollama")

In [13]:
DB="exam.db"
with sqlite3.connect(DB) as conn:
    cursor=conn.cursor()
    cursor.execute("drop table if exists QA")
    cursor.execute("create table if not exists QA (id integer primary key autoincrement, ques text ,ans text)")
    conn.commit()

In [14]:
def insert_QA(ques_ans):
    with sqlite3.connect(DB) as conn:
        cursor=conn.cursor()
        cursor.executemany("insert into QA (ques,ans) values (?,?)",(ques_ans))
        conn.commit()

In [15]:
Ques_Ans = [

    ("Why is Deep Learning powerful for image recognition?",
     "Because deep neural networks can automatically extract important features such as edges, shapes, and objects from images."),

    ("How does Machine Learning differ from traditional programming?",
     "In traditional programming rules are explicitly written, while in Machine Learning the system learns rules from data."),

    ("Why do we split data into training and testing sets?",
     "To evaluate how well the model generalizes to unseen data and avoid overfitting."),

    ("What happens if the learning rate is too high?",
     "The model may fail to converge and can overshoot the optimal solution during training."),

    ("What happens if the learning rate is too low?",
     "Training becomes very slow and the model may get stuck before reaching the optimal solution."),

    ("Why are activation functions important in neural networks?",
     "They introduce non-linearity which allows neural networks to learn complex relationships in data."),

    ("How does CNN reduce the number of parameters compared to fully connected networks?",
     "CNN uses shared filters and local connections which greatly reduce the number of trainable parameters."),

    ("Why are LSTMs better than standard RNNs for long sequences?",
     "Because LSTMs can remember important information for long periods and reduce the vanishing gradient problem."),

    ("What is the purpose of dropout in deep learning?",
     "Dropout reduces overfitting by randomly disabling some neurons during training."),

    ("Why is normalization important in Machine Learning?",
     "Normalization scales features to similar ranges which improves training stability and convergence speed."),

    ("How does backpropagation improve a neural network?",
     "It calculates errors and updates weights to minimize the loss function."),

    ("What is the role of an optimizer in deep learning?",
     "The optimizer updates model parameters to reduce prediction error during training."),

    ("Why is GPU commonly used in Deep Learning?",
     "Because GPUs can process many calculations in parallel which speeds up neural network training."),

    ("How does transfer learning save training time?",
     "It uses knowledge from a pre-trained model so the network does not need to learn from scratch."),

    ("What is the difference between classification and regression?",
     "Classification predicts categories while regression predicts continuous numerical values."),

    ("Why can Deep Learning require large datasets?",
     "Because deep networks have many parameters and need large amounts of data to learn effectively."),

    ("How does data augmentation help in training?",
     "It increases dataset diversity by modifying existing data which improves model generalization."),

    ("What is the vanishing gradient problem?",
     "It occurs when gradients become extremely small during backpropagation making learning difficult in deep networks."),

    ("Why is cross-validation useful in Machine Learning?",
     "It provides a more reliable evaluation of model performance using different subsets of data."),

    ("How does attention mechanism improve deep learning models?",
     "Attention helps the model focus on the most important parts of the input data during prediction.")
]

In [16]:
insert_QA(Ques_Ans)

In [17]:
def get_ques_ans():
    with sqlite3.connect(DB) as conn:
        cursor=conn.cursor()
        #cursor.execute("select * from QA" )
        cursor.execute("select ques,ans from QA order by random() limit 3")
        row=cursor.fetchall()
        return row

In [18]:
get_ques_ans()


[('How does data augmentation help in training?',
  'It increases dataset diversity by modifying existing data which improves model generalization.'),
 ('What is the purpose of dropout in deep learning?',
  'Dropout reduces overfitting by randomly disabling some neurons during training.'),
 ('How does transfer learning save training time?',
  'It uses knowledge from a pre-trained model so the network does not need to learn from scratch.')]

In [19]:
get_ques_ans_tool = [
   {
  "type": "function",
  "function": {
    "name": "get_ques_ans",
    "description": "Fetch a random exam question and its correct answer from the SQL database",
    "parameters": {
      "type": "object",
      "properties": {}
    }
  }
}
]

In [20]:
system_prompt = """
Strict AI Exam Tutor.

RULES:
1. TOOL CALL: Call before every question. If invalid/fails, output: call for anthor one
2. EXACT: Ask tool questions exactly as received. No edits, MCQ, or internal knowledge, display the question only,dont say any thing.
3. NO REPEAT: if the question in the cahting history never ask the question again and call the tool to get anthor one. 
6. FINAL: After Q3 user answer, output ONLY the final numerical average for the three questions (e.g., 8.3) and write "finished your exam".
7. LIMITS: be strict,no explanations, no tool names, no correct answers revealed, if the user ask exam again, never give the exam again.
8. if the student finish his exam print"End of exam".
"""

In [12]:
get_ques_ans_tool

[{'type': 'function',
  'function': {'name': 'get_ques_ans',
   'description': 'Fetch a random exam question and its correct answer from the SQL database',
   'parameters': {'type': 'object', 'properties': {}}}}]

In [13]:
def chat(message,history):
    history=[{"role":h["role"],"content":h["content"] } for h in history]
    messages=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    response=ollama.chat.completions.create(model="gemma4",messages=messages,tools=get_ques_ans_tool)
   # while response.choices[0].message.content=="":
        # response=ollama.chat.completions.create(model= "gemma4",messages=messages+[message]+[response],tools=get_ques_ans_tool)

    while response.choices[0].finish_reason=="tool_calls":
        print("tool_call")
        message=response.choices[0].message
        response=handel_call_tool(message)
        #messages.append(message)
        #messages.append(response)
        response=ollama.chat.completions.create(model= "gemma4",messages=messages+[message]+[response],tools=get_ques_ans_tool)
        while response.choices[0].message.content is None:
            response=ollama.chat.completions.create(model= "gemma4",messages=messages+[message]+[response],tools=get_ques_ans_tool)
 

    print(response.choices[0].message.content)
    return response.choices[0].message.content

In [14]:
def handel_call_tool(message):
    tool_call=message.tool_calls[0]
    print(tool_call)
    if tool_call.function.name=="get_ques_ans":
        ques_ans=get_ques_ans()
        response = {
                "role": "tool",
                "content": json.dumps({
                    "question": ques_ans[0],
                    
                }, ensure_ascii=False),
                "tool_call_id": tool_call.id
            }
        print (response)
        return(response)

In [16]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


tool_call
ChatCompletionMessageFunctionToolCall(id='call_ika2hdpz', function=Function(arguments='{}', name='get_ques_ans'), type='function', index=0)
{'role': 'tool', 'content': '{"question": "What is DSB-AM?"}', 'tool_call_id': 'call_ika2hdpz'}

tool_call
ChatCompletionMessageFunctionToolCall(id='call_5z1bmw3s', function=Function(arguments='{}', name='get_ques_ans'), type='function', index=0)
{'role': 'tool', 'content': '{"question": "Difference between DSB-SC and conventional AM?"}', 'tool_call_id': 'call_5z1bmw3s'}
Difference between DSB-SC and conventional AM?
tool_call
ChatCompletionMessageFunctionToolCall(id='call_3zby7zau', function=Function(arguments='{}', name='get_ques_ans'), type='function', index=0)
{'role': 'tool', 'content': '{"question": "What is DSB-AM?"}', 'tool_call_id': 'call_3zby7zau'}
What is DSB-AM?
tool_call
ChatCompletionMessageFunctionToolCall(id='call_rwzdw39a', function=Function(arguments='{}', name='get_ques_ans'), type='function', index=0)
{'role': 'tool', 

In [48]:
def chat(message,history):
    print(history)
    return  "hi"

In [21]:
num_quest=3
ques_crans=get_ques_ans()   #quetsion and correct answe
ques_id=0
student_answer=[]

In [ ]:
   question[2][0]

'Difference between DSB-SC and conventional AM?'

In [28]:
def chat(message,history):
     questions_data=[]
     global ques_id
     global student_answer
 
     if ques_id<3:
         ques_id=ques_id+1
         student_answer.append(message if ques_id >0 else None)
         return ques_crans[ques_id-1][0]  #return question only 
      
     elif ques_id==3:
        student_answer.append(message if ques_id -1>0 else None)
        for i, element in enumerate(ques_crans):
             questions_data.append({
                "question": element[0],
                "correct_answer": element[1],
                "student_answer": student_answer[i]
            })


             print(questions_data)
            
        prompt = f"""
        Evaluate the following student answers.

        Data:
        {json.dumps(questions_data, indent=2)}

        Return ONLY valid JSON in this format:

        {{
        "results": [
        {{
        "question": "...",
      "score": 0-10,
      "is_correct": true,
      "feedback": "short feedback"
       }}
       ],
       "final_score": 0-10
       }}
       """
        response=ollama.chat.completions.create(model="gemma4",messages=[{"role": "user","content":prompt}])
        ques_id=ques_id+1
        return response.choices[0].message.content
     else:
        return "End of exam"




In [ ]:
def chat(message, history):

    global ques_id
    global student_answer

    questions_data = []

    # save the student answer
    if ques_id > 0:
        student_answer.append(message)

    # ask the next question
    if ques_id < len(ques_crans):

        question = ques_crans[ques_id][0]

        ques_id += 1

        return question

    # get the score
    elif ques_id == len(ques_crans):

        for i, element in enumerate(ques_crans):

            questions_data.append({
                "question": element[0],
                "correct_answer": element[1],
                "student_answer": student_answer[i]
            })

        print(questions_data)

        prompt = f"""
Evaluate the following student answers.

Data:
{json.dumps(questions_data, indent=2)}

Return ONLY valid JSON.

"""

        response = ollama.chat.completions.create(
            model="gemma4",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        ques_id += 1

        return response.choices[0].message.content

    else:
        return "End of exam"

In [25]:
num_quest=3
ques_crans=get_ques_ans()   #quetsion and correct answe
ques_id=0
student_answer=[]
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


[{'question': 'Why is GPU commonly used in Deep Learning?', 'correct_answer': 'Because GPUs can process many calculations in parallel which speeds up neural network training.', 'student_answer': 'i dont know'}, {'question': 'How does backpropagation improve a neural network?', 'correct_answer': 'It calculates errors and updates weights to minimize the loss function.', 'student_answer': 'i dont know'}, {'question': 'What is the difference between classification and regression?', 'correct_answer': 'Classification predicts categories while regression predicts continuous numerical values.', 'student_answer': 'in classification you hve some classes he out put can be one of them, more over the input which related to the class  will make the output 1 at this class and zero other where. in regression you have one output and can take vlue greater than 1 despite classification'}]


In [23]:
def get_score_chat(message,history):
    history=[{"role":h["role"],"content":h["content"] } for h in history]
    messages=[{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    response=ollama.chat.completions.create(model="gemma4",messages=messages,tools=get_ques_ans_tool)
   # while response.choices[0].message.content=="":
        # response=ollama.chat.completions.create(model= "gemma4",messages=messages+[message]+[response],tools=get_ques_ans_tool)


    print(response.choices[0].message.content)
    return response.choices[0].message.content

In [63]:
questions_data=[]

In [61]:
for i, element in enumerate(ques_crans):

  questions_data.append((element[0],element[1],student_answer[i]))

IndexError: list index out of range

In [27]:
for i, element in enumerate(ques_crans):
  questions_data.append [
      {
          "question":element[0],
          "correct_answer":element[1],
          "user_answer":student_answer[i]
      }
    ]
          
questions_data = [
    {
        "question": "What is DSB-SC?",
        "correct_answer": "Double Sideband Suppressed Carrier",
        "user_answer": "AM without carrier"
    },
    {
        "question": "Why coherent detection is required in DSB-SC?",
        "correct_answer": "Because the carrier is suppressed",
        "user_answer": "To recover the signal correctly"
    }
]

prompt = f"""
Evaluate the following student answers.

Data:
{json.dumps(questions_data, indent=2)}

Return ONLY valid JSON in this format:

{{
  "results": [
    {{
      "question": "...",
      "score": 0-10,
      "is_correct": true,
      "feedback": "short feedback"
    }}
  ],
  "final_score": 0-10
}}
"""

NameError: name 'questions_data' is not defined